# Anti-DreamBooth Full Test (A100 40GB)

This notebook runs the complete Anti-DreamBooth pipeline:

1. **Setup** — Install dependencies, clone repo, download SD model
2. **Baseline** — Train DreamBooth on clean images (no defense)
3. **Attack** — Run ASPL to generate adversarial perturbations
4. **Defense** — Train DreamBooth on perturbed images
5. **Inference** — Generate images from both models
6. **Compare** — Visual side-by-side comparison

> **Runtime:** A100 40GB GPU required. Total ~45-60 min.

---
## 0. GPU Check

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory
    print(f"Memory: {total_mem / 1e9:.1f} GB")
    assert total_mem > 35e9, "This notebook requires A100 40GB!"
    print("\n>>> A100 40GB confirmed. Good to go!")
else:
    raise RuntimeError("No GPU detected! Go to Runtime > Change runtime type > A100 GPU")

---
## 1. Install Dependencies & Clone Repo

In [ ]:
%%time
# The original requirements.txt pins ancient versions (torch 1.13.1, diffusers 0.13.1)
# that are unavailable on modern Colab (CUDA 13.0, Python 3.12+).
#
# We use the versions from README's LoRA-tested config, which are
# compatible with both the main code and the LoRA code.
# diffusers 0.23.1 requires peft<=0.6.2.

!pip install -q \
    diffusers==0.23.1 \
    transformers==4.48.3 \
    accelerate==0.33.0 \
    peft==0.6.2 \
    datasets \
    ftfy \
    Jinja2 \
    tqdm \
    tensorboard

# xformers is optional — torch 2.x has built-in SDPA (scaled dot-product attention)
# which diffusers 0.19+ uses automatically. Install only if needed:
# !pip install -q xformers

print("\n>>> Dependencies installed.")

In [ ]:
# Verify installations
import torch, diffusers, transformers, accelerate
print(f"torch:        {torch.__version__}")
print(f"diffusers:    {diffusers.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"CUDA:         {torch.version.cuda}")
print(f"SDPA:         {hasattr(torch.nn.functional, 'scaled_dot_product_attention')}")

In [ ]:
import os
REPO_URL = "https://github.com/superbabiix/anti-dreambooth-fork.git"
REPO_DIR = "/content/Anti-DreamBooth-Fork"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

os.chdir(REPO_DIR)
!pwd
!ls

---
## 2. Download Stable Diffusion 2.1-base

The model is downloaded from HuggingFace and saved to `./stable-diffusion/stable-diffusion-2-1-base`.

In [ ]:
%%time
from diffusers import DiffusionPipeline
import torch, os

SD_MODEL_ID = "stabilityai/stable-diffusion-2-1-base"
SD_LOCAL_PATH = "./stable-diffusion/stable-diffusion-2-1-base"

if not os.path.exists(SD_LOCAL_PATH):
    print(f"Downloading {SD_MODEL_ID} ...")
    pipe = DiffusionPipeline.from_pretrained(SD_MODEL_ID, torch_dtype=torch.float16)
    pipe.save_pretrained(SD_LOCAL_PATH)
    del pipe
    torch.cuda.empty_cache()
    print(f"\n>>> Model saved to {SD_LOCAL_PATH}")
else:
    print(f"Model already exists at {SD_LOCAL_PATH}")

!ls {SD_LOCAL_PATH}

---
## 3. Inspect Sample Data

The repo includes sample data for subject `n000050` with 4 images per set.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

def show_images(directory, title, max_images=4):
    imgs = sorted(Path(directory).glob("*.png"))[:max_images]
    if not imgs:
        imgs = sorted(Path(directory).glob("*.jpg"))[:max_images]
    fig, axes = plt.subplots(1, len(imgs), figsize=(4 * len(imgs), 4))
    if len(imgs) == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight='bold')
    for ax, img_path in zip(axes, imgs):
        ax.imshow(Image.open(img_path))
        ax.set_title(img_path.name, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_images("data/n000050/set_A", "Set A (Clean - Training Reference)")
show_images("data/n000050/set_B", "Set B (Clean - Will Be Perturbed)")

---
## 4. Baseline: Train DreamBooth on Clean Images (No Defense)

This trains a standard DreamBooth model on **clean** images to see what an attacker could produce without any defense.

In [ ]:
%%time

# Combine set_A + set_B as clean training data (what attacker would use)
!mkdir -p data/n000050_clean_all
!cp data/n000050/set_A/* data/n000050_clean_all/
!cp data/n000050/set_B/* data/n000050_clean_all/

!accelerate launch train_dreambooth.py \
  --pretrained_model_name_or_path="./stable-diffusion/stable-diffusion-2-1-base" \
  --train_text_encoder \
  --instance_data_dir="data/n000050_clean_all" \
  --class_data_dir="data/class-person" \
  --output_dir="outputs/BASELINE/n000050_DREAMBOOTH" \
  --with_prior_preservation \
  --prior_loss_weight=1.0 \
  --instance_prompt="a photo of sks person" \
  --class_prompt="a photo of person" \
  --inference_prompt="a photo of sks person;a dslr portrait of sks person" \
  --resolution=512 \
  --train_batch_size=2 \
  --gradient_accumulation_steps=1 \
  --learning_rate=5e-7 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --num_class_images=200 \
  --max_train_steps=1000 \
  --checkpointing_steps=500 \
  --center_crop \
  --mixed_precision=bf16 \
  --prior_generation_precision=bf16 \
  --sample_batch_size=8

print("\n>>> Baseline DreamBooth training complete.")

---
## 5. ASPL Attack: Generate Adversarial Perturbations

This runs the **Alternating Surrogate and Perturbation Learning (ASPL)** attack on Set B images to create subtle adversarial noise.

In [ ]:
%%time

OUTPUT_DIR = "outputs/ASPL/n000050_ADVERSARIAL"

!mkdir -p {OUTPUT_DIR}
!cp -r data/n000050/set_A {OUTPUT_DIR}/image_clean
!cp -r data/n000050/set_B {OUTPUT_DIR}/image_before_adding_noise

!accelerate launch attacks/aspl.py \
  --pretrained_model_name_or_path="./stable-diffusion/stable-diffusion-2-1-base" \
  --instance_data_dir_for_train="data/n000050/set_A" \
  --instance_data_dir_for_adversarial="data/n000050/set_B" \
  --instance_prompt="a photo of sks person" \
  --class_data_dir="data/class-person" \
  --num_class_images=200 \
  --class_prompt="a photo of person" \
  --output_dir="{OUTPUT_DIR}" \
  --center_crop \
  --with_prior_preservation \
  --prior_loss_weight=1.0 \
  --resolution=512 \
  --train_text_encoder \
  --train_batch_size=1 \
  --max_train_steps=50 \
  --max_f_train_steps=3 \
  --max_adv_train_steps=6 \
  --checkpointing_iterations=10 \
  --learning_rate=5e-7 \
  --pgd_alpha=5e-3 \
  --pgd_eps=5e-2

print("\n>>> ASPL attack complete. Perturbed images saved.")

In [ ]:
# Show perturbed images vs originals
print("Perturbed image checkpoints:")
!ls outputs/ASPL/n000050_ADVERSARIAL/noise-ckpt/

show_images("data/n000050/set_B", "Original Set B (Clean)")
show_images("outputs/ASPL/n000050_ADVERSARIAL/noise-ckpt/50", "Set B After ASPL Perturbation (Step 50)")

---
## 6. Defense: Train DreamBooth on Perturbed Images

Now we train DreamBooth on the **perturbed** images. If Anti-DreamBooth works, the generated images should be degraded.

In [ ]:
%%time

!accelerate launch train_dreambooth.py \
  --pretrained_model_name_or_path="./stable-diffusion/stable-diffusion-2-1-base" \
  --train_text_encoder \
  --instance_data_dir="outputs/ASPL/n000050_ADVERSARIAL/noise-ckpt/50" \
  --class_data_dir="data/class-person" \
  --output_dir="outputs/ASPL/n000050_DREAMBOOTH" \
  --with_prior_preservation \
  --prior_loss_weight=1.0 \
  --instance_prompt="a photo of sks person" \
  --class_prompt="a photo of person" \
  --inference_prompt="a photo of sks person;a dslr portrait of sks person" \
  --resolution=512 \
  --train_batch_size=2 \
  --gradient_accumulation_steps=1 \
  --learning_rate=5e-7 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --num_class_images=200 \
  --max_train_steps=1000 \
  --checkpointing_steps=500 \
  --center_crop \
  --mixed_precision=bf16 \
  --prior_generation_precision=bf16 \
  --sample_batch_size=8

print("\n>>> Defense DreamBooth training complete.")

---
## 7. Inference: Generate Images from Both Models

In [ ]:
%%time

# Inference from BASELINE model (trained on clean images)
!python infer.py \
  --model_path="outputs/BASELINE/n000050_DREAMBOOTH/checkpoint-1000" \
  --output_dir="outputs/infer_baseline"

print("\n>>> Baseline inference complete.")

In [ ]:
%%time

# Inference from DEFENDED model (trained on perturbed images)
!python infer.py \
  --model_path="outputs/ASPL/n000050_DREAMBOOTH/checkpoint-1000" \
  --output_dir="outputs/infer_defended"

print("\n>>> Defended inference complete.")

---
## 8. Visual Comparison: Baseline vs Defended

Side-by-side comparison. The **Baseline** (clean) should produce realistic faces. The **Defended** (perturbed) should show degraded/distorted faces — proving Anti-DreamBooth works.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
import os

def get_prompt_dirs(base_dir):
    """Get sorted subdirectories (one per prompt)."""
    return sorted([d for d in Path(base_dir).iterdir() if d.is_dir()])

baseline_dir = "outputs/infer_baseline"
defended_dir = "outputs/infer_defended"

baseline_prompts = get_prompt_dirs(baseline_dir)
defended_prompts = get_prompt_dirs(defended_dir)

n_prompts = min(len(baseline_prompts), len(defended_prompts))
n_samples = 4  # images per prompt to show

for i in range(n_prompts):
    bp = baseline_prompts[i]
    dp = defended_prompts[i]

    b_imgs = sorted(bp.glob("*.png"))[:n_samples]
    d_imgs = sorted(dp.glob("*.png"))[:n_samples]

    if not b_imgs or not d_imgs:
        continue

    cols = max(len(b_imgs), len(d_imgs))
    fig, axes = plt.subplots(2, cols, figsize=(4 * cols, 8))
    fig.suptitle(f"Prompt: {bp.name.replace('_', ' ')}", fontsize=13, fontweight='bold')

    for j in range(cols):
        # Baseline row
        if j < len(b_imgs):
            axes[0][j].imshow(Image.open(b_imgs[j]))
        axes[0][j].axis('off')
        if j == 0:
            axes[0][j].set_ylabel("Baseline\n(No Defense)", fontsize=12, fontweight='bold')

        # Defended row
        if j < len(d_imgs):
            axes[1][j].imshow(Image.open(d_imgs[j]))
        axes[1][j].axis('off')
        if j == 0:
            axes[1][j].set_ylabel("Defended\n(ASPL)", fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

print("\n>>> If Anti-DreamBooth works, the 'Defended' row should show degraded/distorted faces.")

---
## 9. Perturbation Analysis

Visualize how much noise was actually added to the images.

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt

clean_dir = Path("data/n000050/set_B")
noisy_dir = Path("outputs/ASPL/n000050_ADVERSARIAL/noise-ckpt/50")

clean_files = sorted(clean_dir.glob("*.png"))
noisy_files = sorted(noisy_dir.glob("*.png"))

if clean_files and noisy_files:
    n = min(len(clean_files), len(noisy_files))
    fig, axes = plt.subplots(3, n, figsize=(4 * n, 12))
    fig.suptitle("Perturbation Analysis (Clean vs Perturbed vs Amplified Difference)",
                 fontsize=14, fontweight='bold')

    for i in range(n):
        clean_img = np.array(Image.open(clean_files[i]).resize((512, 512))).astype(float)
        noisy_img = np.array(Image.open(noisy_files[i]).resize((512, 512))).astype(float)

        diff = np.abs(noisy_img - clean_img)
        diff_amplified = np.clip(diff * 10, 0, 255).astype(np.uint8)  # 10x amplification

        axes[0][i].imshow(clean_img.astype(np.uint8))
        axes[0][i].set_title("Clean", fontsize=10)
        axes[0][i].axis('off')

        axes[1][i].imshow(noisy_img.astype(np.uint8))
        axes[1][i].set_title("Perturbed", fontsize=10)
        axes[1][i].axis('off')

        axes[2][i].imshow(diff_amplified)
        axes[2][i].set_title(f"Diff x10 (max={diff.max():.1f})", fontsize=10)
        axes[2][i].axis('off')

        # Stats
        print(f"Image {i}: L_inf={diff.max():.2f}, L2={np.sqrt((diff**2).mean()):.2f}, "
              f"Mean={diff.mean():.2f}")

    plt.tight_layout()
    plt.show()
else:
    print("Could not find clean/noisy image pairs. Check paths.")

---
## 10. Summary & Disk Usage

In [ ]:
import os

def dir_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024**3)  # GB

print("=" * 60)
print("ANTI-DREAMBOOTH FULL TEST — SUMMARY")
print("=" * 60)

dirs = {
    "SD 2.1-base model": "./stable-diffusion/stable-diffusion-2-1-base",
    "Baseline DreamBooth": "outputs/BASELINE/n000050_DREAMBOOTH",
    "ASPL attack output": "outputs/ASPL/n000050_ADVERSARIAL",
    "Defended DreamBooth": "outputs/ASPL/n000050_DREAMBOOTH",
    "Baseline inference": "outputs/infer_baseline",
    "Defended inference": "outputs/infer_defended",
}

for name, path in dirs.items():
    if os.path.exists(path):
        size = dir_size(path)
        print(f"  {name:30s} {size:.2f} GB")
    else:
        print(f"  {name:30s} [NOT FOUND]")

print(f"\n  {'TOTAL':30s} {sum(dir_size(p) for p in dirs.values() if os.path.exists(p)):.2f} GB")
print("=" * 60)

# Count generated images
for label, d in [("Baseline", "outputs/infer_baseline"), ("Defended", "outputs/infer_defended")]:
    if os.path.exists(d):
        count = sum(1 for _ in Path(d).rglob("*.png"))
        print(f"  {label} generated images: {count}")

print("\n>>> Done! Compare the baseline vs defended outputs above.")

---
## 11. (Optional) Cleanup

Uncomment to free disk space.

In [ ]:
# Uncomment lines below to clean up
# !rm -rf outputs/
# !rm -rf stable-diffusion/
# !rm -rf data/class-person/
# print("Cleaned up.")